# 04. Camada Gold: Esquema Estrela
Modelo dimensional para responder às perguntas de negócio.
- **fato_vendas**: grão: 1 item de pedido (= 1 unidade vendida). Medidas aditivas: quantidade, valor_venda, valor_frete. Apenas vendas válidas.
- **dim_produto**: o "o quê" (categoria, peso).
- **dim_tempo**: o "quando", com flags de Black Friday e campanhas.
Chaves substitutas (surrogate keys) nas dimensões; PK/FK declaradas no Unity Catalog.

In [0]:
%sql
CREATE OR REPLACE TABLE mvp_olist.gold.dim_tempo
COMMENT 'Dimensão calendário (01/09/2016 a 31/12/2018) com flags de Black Friday e campanhas comerciais'
AS
SELECT
  CAST(date_format(data, 'yyyyMMdd') AS INT) AS sk_data,
  data,
  year(data)                                AS ano,
  quarter(data)                             AS trimestre,
  month(data)                               AS mes,
  date_format(data, 'yyyy-MM')              AS ano_mes,
  dayofweek(data)                           AS dia_semana,
  dayofweek(data) IN (1, 7)                 AS is_fim_de_semana,
  data IN (DATE'2016-11-25', DATE'2017-11-24') AS is_black_friday,
  CASE
    WHEN data BETWEEN DATE'2016-11-25' AND DATE'2016-11-28'
      OR data BETWEEN DATE'2017-11-24' AND DATE'2017-11-27' THEN 'Black Friday'
    WHEN data BETWEEN DATE'2017-05-01' AND DATE'2017-05-14'
      OR data BETWEEN DATE'2018-05-01' AND DATE'2018-05-13' THEN 'Dia das Mães'
    ELSE 'Sem campanha'
  END AS campanha
FROM (SELECT explode(sequence(DATE'2016-09-01', DATE'2018-12-31', INTERVAL 1 DAY)) AS data);

num_affected_rows,num_inserted_rows


In [0]:
%sql
CREATE OR REPLACE TABLE mvp_olist.gold.dim_produto
COMMENT 'Dimensão produto: categoria (PT/EN) e atributos físicos'
AS
SELECT
  CAST(ROW_NUMBER() OVER (ORDER BY product_id) AS INT) AS sk_produto,
  product_id,
  categoria_pt,
  categoria_en,
  peso_g
FROM mvp_olist.silver.products;

num_affected_rows,num_inserted_rows


In [0]:
%sql
CREATE OR REPLACE TABLE mvp_olist.gold.fato_vendas
COMMENT 'Fato de vendas no grão de 1 item de pedido (1 unidade). Apenas vendas válidas (exclui canceled e unavailable)'
AS
SELECT
  CAST(date_format(o.order_purchase_ts, 'yyyyMMdd') AS INT) AS sk_data,
  p.sk_produto,
  i.order_id,
  i.order_item_id,
  1                AS quantidade,
  i.price          AS valor_venda,
  i.freight_value  AS valor_frete
FROM mvp_olist.silver.order_items i
JOIN mvp_olist.silver.orders o    ON i.order_id = o.order_id
JOIN mvp_olist.gold.dim_produto p ON i.product_id = p.product_id
WHERE o.is_venda_valida;

num_affected_rows,num_inserted_rows


In [0]:
%sql
ALTER TABLE mvp_olist.gold.dim_tempo   ALTER COLUMN sk_data    SET NOT NULL;
ALTER TABLE mvp_olist.gold.dim_tempo   ADD CONSTRAINT pk_dim_tempo   PRIMARY KEY (sk_data);
ALTER TABLE mvp_olist.gold.dim_produto ALTER COLUMN sk_produto SET NOT NULL;
ALTER TABLE mvp_olist.gold.dim_produto ADD CONSTRAINT pk_dim_produto PRIMARY KEY (sk_produto);
ALTER TABLE mvp_olist.gold.fato_vendas ADD CONSTRAINT fk_fato_tempo   FOREIGN KEY (sk_data)    REFERENCES mvp_olist.gold.dim_tempo;
ALTER TABLE mvp_olist.gold.fato_vendas ADD CONSTRAINT fk_fato_produto FOREIGN KEY (sk_produto) REFERENCES mvp_olist.gold.dim_produto;

In [0]:
%sql
SELECT t.ano_mes,
       COUNT(DISTINCT f.order_id) AS pedidos,
       SUM(f.quantidade)          AS unidades,
       ROUND(SUM(f.valor_venda), 2) AS receita
FROM mvp_olist.gold.fato_vendas f
JOIN mvp_olist.gold.dim_tempo t ON f.sk_data = t.sk_data
GROUP BY t.ano_mes
ORDER BY t.ano_mes;

ano_mes,pedidos,unidades,receita
2016-09,2,5,207.86
2016-10,290,342,44507.30
2016-12,1,1,10.90
2017-01,787,953,120098.27
2017-02,1718,1936,244959.35
2017-03,2617,2975,368341.32
2017-04,2377,2660,353842.98
2017-05,3640,4106,503159.19
2017-06,3205,3571,429916.61
2017-07,3946,4493,492287.30


In [0]:
%sql
-- dim_tempo
ALTER TABLE mvp_olist.gold.dim_tempo ALTER COLUMN sk_data          COMMENT 'Chave substituta da data no formato AAAAMMDD. PK. Gerada no pipeline';
ALTER TABLE mvp_olist.gold.dim_tempo ALTER COLUMN data             COMMENT 'Data do calendário. Domínio: 2016-09-01 a 2018-12-31. Gerada via sequence()';
ALTER TABLE mvp_olist.gold.dim_tempo ALTER COLUMN ano              COMMENT 'Ano. Domínio: 2016 a 2018';
ALTER TABLE mvp_olist.gold.dim_tempo ALTER COLUMN trimestre        COMMENT 'Trimestre do ano. Domínio: 1 a 4';
ALTER TABLE mvp_olist.gold.dim_tempo ALTER COLUMN mes              COMMENT 'Mês do ano. Domínio: 1 a 12';
ALTER TABLE mvp_olist.gold.dim_tempo ALTER COLUMN ano_mes          COMMENT 'Ano-mês no formato AAAA-MM, para agregações mensais';
ALTER TABLE mvp_olist.gold.dim_tempo ALTER COLUMN dia_semana       COMMENT 'Dia da semana. Domínio: 1 (domingo) a 7 (sábado)';
ALTER TABLE mvp_olist.gold.dim_tempo ALTER COLUMN is_fim_de_semana COMMENT 'Verdadeiro se sábado ou domingo';
ALTER TABLE mvp_olist.gold.dim_tempo ALTER COLUMN is_black_friday  COMMENT 'Verdadeiro no dia da Black Friday (25/11/2016 e 24/11/2017)';
ALTER TABLE mvp_olist.gold.dim_tempo ALTER COLUMN campanha         COMMENT 'Campanha comercial. Domínio: Black Friday (sex a seg), Dia das Mães (2 semanas antes), Sem campanha';

-- dim_produto
ALTER TABLE mvp_olist.gold.dim_produto ALTER COLUMN sk_produto   COMMENT 'Chave substituta do produto. PK. Gerada via ROW_NUMBER()';
ALTER TABLE mvp_olist.gold.dim_produto ALTER COLUMN product_id   COMMENT 'Identificador original do produto (chave natural). Linhagem: bronze.products.product_id';
ALTER TABLE mvp_olist.gold.dim_produto ALTER COLUMN categoria_pt COMMENT 'Categoria em português; nulos preenchidos com sem_categoria. Linhagem: bronze.products';
ALTER TABLE mvp_olist.gold.dim_produto ALTER COLUMN categoria_en COMMENT 'Categoria em inglês; fallback para português quando sem tradução. Linhagem: JOIN bronze.products x bronze.category_translation';
ALTER TABLE mvp_olist.gold.dim_produto ALTER COLUMN peso_g       COMMENT 'Peso do produto em gramas. Pode ser nulo (2 produtos). Linhagem: bronze.products.product_weight_g';

-- fato_vendas
ALTER TABLE mvp_olist.gold.fato_vendas ALTER COLUMN sk_data       COMMENT 'FK para dim_tempo: data da compra. Linhagem: silver.orders.order_purchase_ts';
ALTER TABLE mvp_olist.gold.fato_vendas ALTER COLUMN sk_produto    COMMENT 'FK para dim_produto';
ALTER TABLE mvp_olist.gold.fato_vendas ALTER COLUMN order_id      COMMENT 'Dimensão degenerada: identificador do pedido';
ALTER TABLE mvp_olist.gold.fato_vendas ALTER COLUMN order_item_id COMMENT 'Dimensão degenerada: sequencial do item no pedido (1 a N)';
ALTER TABLE mvp_olist.gold.fato_vendas ALTER COLUMN quantidade    COMMENT 'Unidades vendidas; sempre 1 (cada item = 1 unidade na fonte). Medida aditiva';
ALTER TABLE mvp_olist.gold.fato_vendas ALTER COLUMN valor_venda   COMMENT 'Preço do item em R$. Domínio: > 0 (mín 0,85; máx 6.735). Medida aditiva. Linhagem: silver.order_items.price';
ALTER TABLE mvp_olist.gold.fato_vendas ALTER COLUMN valor_frete   COMMENT 'Frete do item em R$. Domínio: >= 0 (0 = frete grátis). Medida aditiva. Linhagem: silver.order_items.freight_value';

In [0]:
%sql
DESCRIBE TABLE EXTENDED mvp_olist.gold.fato_vendas;

col_name,data_type,comment
sk_data,int,FK para dim_tempo: data da compra. Linhagem: silver.orders.order_purchase_ts
sk_produto,int,FK para dim_produto
order_id,string,Dimensão degenerada: identificador do pedido
order_item_id,int,Dimensão degenerada: sequencial do item no pedido (1 a N)
quantidade,int,Unidades vendidas; sempre 1 (cada item = 1 unidade na fonte). Medida aditiva
valor_venda,"decimal(10,2)","Preço do item em R$. Domínio: > 0 (mín 0,85; máx 6.735). Medida aditiva. Linhagem: silver.order_items.price"
valor_frete,"decimal(10,2)",Frete do item em R$. Domínio: >= 0 (0 = frete grátis). Medida aditiva. Linhagem: silver.order_items.freight_value
,,
# Delta Statistics Columns,,
Column Names,"order_id, order_item_id, valor_frete, valor_venda, sk_data, quantidade, sk_produto",
